# 02. Feature Engineering & Leak-Free Dataset Preprocessing

**Research Paper:** *Robust Federated Learning-Based Intrusion Detection for Heterogeneous Vehicular CAN Networks*  
**Objective:** Transform raw automotive CAN frames into standardized sliding sequence windows for 1D-CNN baseline training without temporal data leakage.

---

### Pipeline Steps:
1. Ingest raw CAN logs across attack types and normal operations.
2. Parse hexadecimal CAN IDs ($0	ext{x}000 - 0	ext{x}7	ext{FF}$) and scale to $[0, 1]$.
3. Standardize DLC ($0 - 8$) and 8 payload bytes ($D_0 \dots D_7 \in [0, 255]$ scaled to $[0, 1]$).
4. Compute inter-arrival time $\Delta t$ with log scaling.
5. Construct sequential sliding windows ($W=16$ messages per sample, stride $S=16$).
6. Apply strict **chronological time-series splitting** ($70\%$ Train, $15\%$ Validation, $15\%$ Test).
7. Save processed datasets under `data/processed/` and splits under `data/splits/` with full metadata JSON.


In [1]:
# Setup & Imports
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import discover_raw_datasets, load_csv_dataset, load_txt_normal
from src.preprocessing.pipeline import CANPreprocessor, create_sliding_windows, chronological_split

SEED = 42
np.random.seed(SEED)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_SPLITS = PROJECT_ROOT / "data" / "splits"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_SPLITS.mkdir(parents=True, exist_ok=True)

print("Preprocessing directories initialized.")


Preprocessing directories initialized.


In [2]:
# 1. Load Raw CAN Datasets for Combined Benchmark
raw_files = discover_raw_datasets(DATA_RAW)
print(f"Available raw datasets: {list(raw_files.keys())}")

scenario_dfs = {}
N_FRAMES_PER_FILE = 100000

for name, path in raw_files.items():
    print(f"Loading {name} ({path.name})...")
    if path.suffix == ".txt":
        scenario_dfs[name] = load_txt_normal(path, nrows=N_FRAMES_PER_FILE)
    else:
        scenario_dfs[name] = load_csv_dataset(path, nrows=N_FRAMES_PER_FILE, attack_name=name)

print("\nLoaded all scenarios successfully.")


Available raw datasets: ['DoS', 'Fuzzy', 'Gear', 'RPM', 'Normal']
Loading DoS (DoS_dataset.csv)...


Loading Fuzzy (Fuzzy_dataset.csv)...


Loading Gear (gear_dataset.csv)...


Loading RPM (RPM_dataset.csv)...


Loading Normal (normal_run_data.txt)...



Loaded all scenarios successfully.


In [3]:
# 2. Extract Features, Sliding Windows & Chronological Split Per Scenario
from src.preprocessing.pipeline import process_and_split_scenarios

WINDOW_SIZE = 16
STEP_SIZE = 16  # Non-overlapping contiguous windows

print(f"Processing and splitting per scenario (Window Size = {WINDOW_SIZE}, Step = {STEP_SIZE})...")
preprocessor = CANPreprocessor()

unified_splits = process_and_split_scenarios(
    scenario_dfs=scenario_dfs,
    preprocessor=preprocessor,
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15
)

X_train, y_train = unified_splits["X_train"], unified_splits["y_train"]
X_val, y_val = unified_splits["X_val"], unified_splits["y_val"]
X_test, y_test = unified_splits["X_test"], unified_splits["y_test"]

print("\n=== Chronological Partition Summary (Leak-Free) ===")
print(f"Train Set: {X_train.shape[0]:,d} windows | Attack: {y_train.sum():,d} ({y_train.mean()*100:.1f}%), Normal: {(len(y_train)-y_train.sum()):,d}")
print(f"Val Set:   {X_val.shape[0]:,d} windows | Attack: {y_val.sum():,d} ({y_val.mean()*100:.1f}%), Normal: {(len(y_val)-y_val.sum()):,d}")
print(f"Test Set:  {X_test.shape[0]:,d} windows | Attack: {y_test.sum():,d} ({y_test.mean()*100:.1f}%), Normal: {(len(y_test)-y_test.sum()):,d}")


Processing and splitting per scenario (Window Size = 16, Step = 16)...



=== Chronological Partition Summary (Leak-Free) ===
Train Set: 21,875 windows | Attack: 8,553 (39.1%), Normal: 13,322
Val Set:   4,685 windows | Attack: 1,798 (38.4%), Normal: 2,887
Test Set:  4,690 windows | Attack: 1,790 (38.2%), Normal: 2,900


In [4]:
# 3. Persist Processed Datasets, Partition Splits, and Metadata
out_npz_path = DATA_PROCESSED / "can_ids_processed_dataset.npz"
np.savez_compressed(
    out_npz_path,
    X_train=X_train, y_train=y_train, y_types_train=unified_splits["y_types_train"],
    X_val=X_val, y_val=y_val, y_types_val=unified_splits["y_types_val"],
    X_test=X_test, y_test=y_test, y_types_test=unified_splits["y_types_test"]
)
print(f"Saved processed dataset: {out_npz_path} ({out_npz_path.stat().st_size / (1024*1024):.2f} MB)")

# Persist partition splits to data/splits/
np.savez_compressed(DATA_SPLITS / "train_split.npz", X=X_train, y=y_train, y_types=unified_splits["y_types_train"])
np.savez_compressed(DATA_SPLITS / "val_split.npz", X=X_val, y=y_val, y_types=unified_splits["y_types_val"])
np.savez_compressed(DATA_SPLITS / "test_split.npz", X=X_test, y=y_test, y_types=unified_splits["y_types_test"])
print(f"Saved partition splits under: {DATA_SPLITS}")

# Save metadata
total_frames = sum(len(df) for df in scenario_dfs.values())
metadata = {
    "feature_names": preprocessor.feature_names,
    "window_size": WINDOW_SIZE,
    "step_size": STEP_SIZE,
    "total_raw_frames": total_frames,
    "total_windows": len(X_train) + len(X_val) + len(X_test),
    "split_ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
    "split_counts": {
        "train": int(len(X_train)),
        "val": int(len(X_val)),
        "test": int(len(X_test))
    },
    "class_counts": {
        "train_attack": int(y_train.sum()),
        "train_normal": int(len(y_train) - y_train.sum()),
        "val_attack": int(y_val.sum()),
        "val_normal": int(len(y_val) - y_val.sum()),
        "test_attack": int(y_test.sum()),
        "test_normal": int(len(y_test) - y_test.sum())
    },
    "scenario_statistics": unified_splits["scenario_stats"]
}

meta_path = DATA_PROCESSED / "preprocessing_metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved preprocessing metadata: {meta_path}")


Saved processed dataset: C:\Users\Kratik\IIIT Academics\Hustle\Major_Project_CAN_IDS\data\processed\can_ids_processed_dataset.npz (3.08 MB)


Saved partition splits under: C:\Users\Kratik\IIIT Academics\Hustle\Major_Project_CAN_IDS\data\splits
Saved preprocessing metadata: C:\Users\Kratik\IIIT Academics\Hustle\Major_Project_CAN_IDS\data\processed\preprocessing_metadata.json
